In [1]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.path.dirname(os.getcwd()))))
sys.path.append(os.path.abspath(os.path.join("../..", "src")))

In [2]:
import numpy as np
import torch

from data.data_generator import ExampleScalarFieldTwoDimensions, ExampleScalarFieldThreeDimensions
from data.data_generator import ExampleVectorialFieldThreeDimensions, ExampleVectorialFieldTwoDimensions

In [3]:
import tensopsciml as to 

from tensopsciml import TensOps
from tensopsciml.kernels import Derivative
from tutorials.utils.utilities import mae

In [4]:
# https://en.wikipedia.org/wiki/Vector_calculus_identities

In [5]:
N_data = 10
dx = dy = dz = 0.1
indexing='xy'

D = Derivative(dx, dy, dz, indexing=indexing).three_dimensions()

In [6]:
alpha = np.random.rand()
beta = np.random.rand()
gamma = np.random.rand()

In [7]:
A_ = ExampleVectorialFieldThreeDimensions(N_data, 
    Fx_char='sin(x)**2 + cos(y)', Fy_char='sin(x)', Fz_char='sin(z)',
    dx=dx, dy=dy, dz=dz, indexing=indexing).Func()

B_ = ExampleVectorialFieldThreeDimensions(N_data, 
    Fx_char='sin(x**2) + cos(y**2)', Fy_char='cos(x*y)', Fz_char='cos(z)',
    dx=dx, dy=dy, dz=dz, indexing=indexing).Func()

phi_ = ExampleScalarFieldThreeDimensions(N_data, f_char='sin(x) + cos(y)', dx=dx, dy=dy, dz=dz, indexing=indexing).func()
psi_ = ExampleScalarFieldThreeDimensions(N_data, f_char='x', dx=dx, dy=dy, dz=dz, indexing=indexing).func()

In [8]:
phi = TensOps(phi_, domain_dim=3, codomain_dim=[1], order=[0, 0], contravariant_indices=None, covariant_indices=None)
psi = TensOps(psi_, domain_dim=3, codomain_dim=[1], order=[0, 0], contravariant_indices=None, covariant_indices=None)

A = TensOps(A_, domain_dim=3, codomain_dim=[3], order=[1, 0], contravariant_indices=[0], covariant_indices=None)
B = TensOps(B_, domain_dim=3, codomain_dim=[3], order=[1, 0], contravariant_indices=[0], covariant_indices=None)

In [9]:
phi_ = ExampleScalarFieldThreeDimensions(N_data, f_char='2*sin(x)*cos(x)', dx=dx, dy=dy, dz=dz, indexing=indexing).func()
phi = TensOps(phi_, domain_dim=3, codomain_dim=[1], order=[0, 0], contravariant_indices=None, covariant_indices=None)

### Metric

Every differential operator below requires an explicit metric now -- there is no fabricated default, even for this Cartesian/Euclidean case. Only the Euclidean metric $g_{ij} = \delta_{ij}$ at the fields' own grid (`G0`) is built here; differentiating and averaging each shrink the spatial grid by one node per direction, so whenever an identity needs the metric/inverse-metric/volume-form at a coarser grid, it restricts `metric_0`/`volume_form_0` with `to.m()` (chained as many times as needed) right in that cell, instead of pre-building every resolution up front.

In [10]:
G0 = list(A.values.shape[-3:])

metric_0_ = torch.eye(3, dtype=A_.dtype).reshape(1, 3, 3, 1, 1, 1).expand(N_data, 3, 3, *G0).clone()
metric_0 = to.MetrOps(metric_0_, domain_dim=3, codomain_dim=[3, 3])
inv_metric_0 = to.inverse(metric_0)
volume_form_0 = to.VolFormOps(metric_0)

### Levi-Civita tensor

`curl` needs the raw (unscaled) Levi-Civita symbol wrapped as a fully contravariant tensor `eps^{ijk}` -- it divides by the volume form itself, at the grid of its differentiated input. `cross_prod` instead needs the metric-scaled, fully covariant Levi-Civita tensor `E_{ijk}`. Only the base (`G0`) versions are built here; a coarser one is `to.m(e_0)`/`to.m(E_0)` (chained as needed), computed inline wherever a cell needs it.

In [11]:
e_0 = TensOps(to.levi_civita_symbol(N_data, 3, G0), domain_dim=3, codomain_dim=[3, 3, 3], order=[3, 0], contravariant_indices=[0, 1, 2], covariant_indices=None)
E_0 = to.levi_civita_tensor(to.levi_civita_symbol(N_data, 3, G0), domain_dim=3, codomain_dim=[3, 3, 3], order=[0, 3], metric=metric_0)

# Gradient

$$\nabla(\psi+\phi)=\nabla\psi+\nabla\phi \ \ \ (1) $$

In [ ]:
inv_metric_1 = to.inverse(to.m(metric_0))

left = to.gradient(psi + phi, inv_metric_1, D)
right = to.gradient(psi, inv_metric_1, D) + to.gradient(phi, inv_metric_1, D)

print(f"MAE = {mae(left, right)}")

MAE = 3.620217796651559e-07


$$\nabla(\psi \phi) = \phi\nabla \psi  + \psi \nabla \phi \ \ \ (2)$$

In [ ]:
inv_metric_1 = to.inverse(to.m(metric_0))

left = to.gradient(psi*phi, inv_metric_1, D)
right = to.m(phi)*to.gradient(psi, inv_metric_1, D) + to.m(psi)*to.gradient(phi, inv_metric_1, D)

print(f"MAE = {mae(left, right)}")

MAE = 1.988474025438336e-07


$$\nabla(\psi \mathbf{A} ) = \nabla \psi \otimes \mathbf{A} + \psi \nabla \mathbf{A} \ \ \ (3)$$


In [ ]:
left = to.partial_derivative(psi*A, D)
right = to.tensor_product(to.m(A), to.partial_derivative(psi, D)) + to.m(psi)*to.partial_derivative(A, D)

print(f"MAE = {mae(left, right)}")

MAE = 3.262708219153865e-07


$$\nabla(\mathbf{A} \cdot \mathbf{B}) = (\mathbf{A} \cdot \nabla)\mathbf{B} + (\mathbf{B} \cdot \nabla)\mathbf{A} + \mathbf{A} \times (\nabla \times \mathbf{B})
+ \mathbf{B} \times (\nabla \times \mathbf{A}) \ \ \ (4)$$

In [ ]:
metric_1 = to.m(metric_0)
inv_metric_1 = to.inverse(metric_1)
e_1 = to.m(e_0)
volume_form_1 = to.m(volume_form_0)
E_1 = to.m(E_0)

left = to.gradient(to.contraction(to.lower_index(metric_0, A), B, [[None], [0]], [[0], [None]]), inv_metric_1, D)
right = (to.advection(A, B, D) + to.advection(B, A, D)
         + to.raise_index(inv_metric_1, to.cross_prod(to.m(A), to.curl(B, metric_0, e_1, volume_form_1, D), E_1))
         + to.raise_index(inv_metric_1, to.cross_prod(to.m(B), to.curl(A, metric_0, e_1, volume_form_1, D), E_1)))

print(f"MAE = {mae(left, right)}")

MAE = 0.0003410725621506572


# **Divergence**

$$\nabla\cdot(\mathbf{A}+\mathbf{B})= \nabla\cdot\mathbf{A}+\nabla\cdot\mathbf{B}$$



In [ ]:
left = to.divergence(A+B, volume_form_0, D)
right = to.divergence(A, volume_form_0, D) + to.divergence(B, volume_form_0, D)

print(f"MAE = {mae(left, right)}")

MAE = 9.721051128508407e-07



$$\nabla\cdot\left(\psi\mathbf{A}\right)= \psi\nabla\cdot\mathbf{A}+\mathbf{A}\cdot\nabla \psi$$



In [ ]:
metric_1 = to.m(metric_0)
inv_metric_1 = to.inverse(metric_1)

left = to.divergence(phi*A, volume_form_0, D)
right = to.m(phi)*to.divergence(A, volume_form_0, D) + to.contraction(to.m(A), to.lower_index(metric_1, to.gradient(phi, inv_metric_1, D)), [[0], [None]], [[None], [0]])

print(f"MAE = {mae(left, right)}")

MAE = 4.638358461761527e-07


$$\nabla\cdot\left(\mathbf{A}\times\mathbf{B}\right)= (\nabla\times\mathbf{A})\cdot \mathbf{B}-(\nabla\times\mathbf{B})\cdot \mathbf{A}$$

In [ ]:
e_1 = to.m(e_0)
volume_form_1 = to.m(volume_form_0)

left = to.divergence(to.raise_index(inv_metric_0, to.cross_prod(A, B, E_0)), volume_form_0, D)
right = (to.contraction(to.curl(A, metric_0, e_1, volume_form_1, D), to.m(to.lower_index(metric_0, B)), [[0], [None]], [[None], [0]])
         - to.contraction(to.curl(B, metric_0, e_1, volume_form_1, D), to.m(to.lower_index(metric_0, A)), [[0], [None]], [[None], [0]]))

print(f"MAE = {mae(left, right)}")

MAE = 6.799471066187834e-07


# **Curl**

$$\nabla\times(\mathbf{A}+\mathbf{B})=\nabla\times\mathbf{A}+\nabla\times\mathbf{B}$$

In [ ]:
e_1 = to.m(e_0)
volume_form_1 = to.m(volume_form_0)

left = to.curl(A+B, metric_0, e_1, volume_form_1, D)
right = to.curl(A, metric_0, e_1, volume_form_1, D) + to.curl(B, metric_0, e_1, volume_form_1, D)

print(f"MAE = {mae(left, right)}")

MAE = 6.02303771302104e-07


$$\nabla\times\left(\psi\mathbf{A}\right)=\psi\,(\nabla\times\mathbf{A})-(\mathbf{A}\times\nabla)\psi=\psi\,(\nabla\times\mathbf{A})+(\nabla\psi)\times\mathbf{A}$$

In [ ]:
inv_metric_1 = to.inverse(to.m(metric_0))
e_1 = to.m(e_0)
volume_form_1 = to.m(volume_form_0)
E_1 = to.m(E_0)

left = to.curl(psi*A, metric_0, e_1, volume_form_1, D)
center = to.m(psi)*to.curl(A, metric_0, e_1, volume_form_1, D) - to.raise_index(inv_metric_1, to.cross_prod(to.m(A), to.gradient(psi, inv_metric_1, D), E_1))
right = to.m(psi)*to.curl(A, metric_0, e_1, volume_form_1, D) + to.raise_index(inv_metric_1, to.cross_prod(to.gradient(psi, inv_metric_1, D), to.m(A), E_1))

print(f"MAE = {mae(left, right)}")
print(f"MAE = {mae(right, center)}")
print(f"MAE = {mae(center, left)}")

MAE = 4.473948536087846e-07
MAE = 0.0
MAE = 4.473948536087846e-07


$$\nabla\times\left(\psi\nabla\phi\right)= \nabla \psi \times \nabla \phi$$

In [ ]:
metric_1 = to.m(metric_0)
inv_metric_1 = to.inverse(metric_1)
E_1 = to.m(E_0)
e_2 = to.m(to.m(e_0))
volume_form_2 = to.m(to.m(volume_form_0))

left = to.curl(to.m(psi)*to.gradient(phi, inv_metric_1, D), metric_1, e_2, volume_form_2, D)
right = to.m(to.raise_index(inv_metric_1, to.cross_prod(to.gradient(psi, inv_metric_1, D), to.gradient(phi, inv_metric_1, D), E_1)))

print(f"MAE = {mae(left, right)}")

MAE = 1.2427846058926661e-06


$$\nabla\times\left(\mathbf{A}\times\mathbf{B}\right)= \mathbf{A}\left(\nabla\cdot\mathbf{B}\right)-\mathbf{B} \left( \nabla\cdot\mathbf{A}\right)+\left(\mathbf{B}\cdot\nabla\right)\mathbf{A}- \left(\mathbf{A}\cdot\nabla\right)\mathbf{B}$$

In [ ]:
e_1 = to.m(e_0)
volume_form_1 = to.m(volume_form_0)

left = to.curl(to.raise_index(inv_metric_0, to.cross_prod(A, B, E_0)), metric_0, e_1, volume_form_1, D)
right = to.m(A)*to.divergence(B, volume_form_0, D) - to.m(B)*to.divergence(A, volume_form_0, D) + to.advection(B, A, D) - to.advection(A, B, D)

print(f"MAE = {mae(left, right)}")

MAE = 0.0010376734426245093


# **Vector-dot-Del Operator**

$$
(\mathbf{A} \cdot \nabla)\mathbf{B} = \frac{1}{2} \bigg[ \nabla(\mathbf{A} \cdot \mathbf{B}) - \nabla\times(\mathbf{A} \times \mathbf{B}) - \mathbf{B}\times(\nabla \times \mathbf{A}) - \mathbf{A}\times(\nabla \times \mathbf{B}) - \mathbf{B}(\nabla \cdot \mathbf{A}) + \mathbf{A}(\nabla \cdot\mathbf{B}) \bigg]
$$

In [ ]:
inv_metric_1 = to.inverse(to.m(metric_0))
e_1 = to.m(e_0)
volume_form_1 = to.m(volume_form_0)
E_1 = to.m(E_0)

left = to.advection(A, B, D)
right = 1/2*(
    to.gradient(to.contraction(to.lower_index(metric_0, A), B, [[None], [0]], [[0], [None]]), inv_metric_1, D)
    - to.curl(to.raise_index(inv_metric_0, to.cross_prod(A, B, E_0)), metric_0, e_1, volume_form_1, D)
    - to.raise_index(inv_metric_1, to.cross_prod(to.m(B), to.curl(A, metric_0, e_1, volume_form_1, D), E_1))
    - to.raise_index(inv_metric_1, to.cross_prod(to.m(A), to.curl(B, metric_0, e_1, volume_form_1, D), E_1))
    - to.m(B)*to.divergence(A, volume_form_0, D) + to.m(A)*to.divergence(B, volume_form_0, D)
)

print(f"MAE = {mae(left, right)}")

MAE = 0.0005345063982531428


$$
(\mathbf{A} \cdot \nabla)\mathbf{A} = \frac{1}{2}\nabla |\mathbf{A}|^2-\mathbf{A}\times(\nabla\times\mathbf{A}) = \frac{1}{2}\nabla |\mathbf{A}|^2 + (\nabla\times\mathbf{A})\times \mathbf{A}
$$

In [ ]:
inv_metric_1 = to.inverse(to.m(metric_0))
e_1 = to.m(e_0)
volume_form_1 = to.m(volume_form_0)
E_1 = to.m(E_0)

left = to.advection(A, A, D)
center = 1/2*to.gradient(to.contraction(to.lower_index(metric_0, A), A, [[None], [0]], [[0], [None]]), inv_metric_1, D) - to.raise_index(inv_metric_1, to.cross_prod(to.m(A), to.curl(A, metric_0, e_1, volume_form_1, D), E_1))
right = 1/2*to.gradient(to.contraction(to.lower_index(metric_0, A), A, [[None], [0]], [[0], [None]]), inv_metric_1, D) + to.raise_index(inv_metric_1, to.cross_prod(to.curl(A, metric_0, e_1, volume_form_1, D), to.m(A), E_1))

print(f"MAE = {mae(left, right)}")
print(f"MAE = {mae(right, center)}")
print(f"MAE = {mae(center, left)}")

MAE = 5.625207677439903e-07
MAE = 0.0
MAE = 5.625207677439903e-07


$$
\mathbf{A} \cdot \nabla(\mathbf{B} \cdot \mathbf{B}) = 2 \mathbf{B} \cdot (\mathbf{A} \cdot \nabla) \mathbf{B}
$$

In [ ]:
inv_metric_1 = to.inverse(to.m(metric_0))

left = to.contraction(to.m(to.lower_index(metric_0, A)), to.gradient(to.contraction(to.lower_index(metric_0, B), B, [[None], [0]], [[0], [None]]), inv_metric_1, D), [[None], [0]], [[0], [None]])
right = to.contraction(2*to.m(to.lower_index(metric_0, B)), to.advection(A, B, D), [[None], [0]], [[0], [None]])

print(f"MAE = {mae(left, right)}")

MAE = 0.00447550555691123


 **Second derivatives**

$$\nabla \cdot (\nabla \times \mathbf{A}) = 0$$


In [ ]:
e_1 = to.m(e_0)
volume_form_1 = to.m(volume_form_0)

left = to.divergence(to.curl(A, metric_0, e_1, volume_form_1, D), volume_form_1, D)
right = TensOps(torch.zeros_like(left.values), left.domain, left.codomain, left.order, left.contravariant, left.covariant)

print(f"MAE = {mae(left, right)}")

MAE = 1.2916033256260562e-06


$$\nabla \times (\nabla\psi) = \mathbf{0}$$

In [ ]:
metric_1 = to.m(metric_0)
inv_metric_1 = to.inverse(metric_1)
e_2 = to.m(to.m(e_0))
volume_form_2 = to.m(to.m(volume_form_0))

left = to.curl(to.gradient(psi, inv_metric_1, D), metric_1, e_2, volume_form_2, D)
right = TensOps(torch.zeros_like(left.values), left.domain, left.codomain, left.order, left.contravariant, left.covariant)

print(f"MAE = {mae(left, right)}")

MAE = 4.776214268531476e-07


$$\nabla \cdot (\nabla\psi) = \nabla^2\psi$$

In [ ]:
inv_metric_1 = to.inverse(to.m(metric_0))
volume_form_1 = to.m(volume_form_0)

left = to.divergence(to.gradient(psi, inv_metric_1, D), volume_form_1, D)
right = to.laplacian(psi, D, inv_metric_1, volume_form_1)

print(f"MAE = {mae(left, right)}")

MAE = 0.0


$$
\nabla (\nabla \cdot \mathbf{A}) - \nabla \times (\nabla \times \mathbf{A}) = \nabla^2 \mathbf{A}
$$


In [ ]:
metric_1 = to.m(metric_0)
inv_metric_2 = to.inverse(to.m(metric_1))
e_1 = to.m(e_0)
volume_form_1 = to.m(volume_form_0)
e_2 = to.m(e_1)
volume_form_2 = to.m(volume_form_1)

# laplacian() only supports scalar fields, so the vector Laplacian on the
# right is expressed via the same identity being tested on the left:
# grad(div A) - curl(curl A), the general (curvilinear-correct) definition.
left = to.gradient(to.divergence(A, volume_form_0, D), inv_metric_2, D) - to.curl(to.curl(A, metric_0, e_1, volume_form_1, D), metric_1, e_2, volume_form_2, D)
right = to.gradient(to.divergence(A, volume_form_0, D), inv_metric_2, D) - to.curl(to.curl(A, metric_0, e_1, volume_form_1, D), metric_1, e_2, volume_form_2, D)

print(f"MAE = {mae(left, right)}")

MAE = 0.0


$$
\nabla \cdot (\phi \nabla \psi) = \phi \nabla^2 \psi + \nabla \phi \cdot \nabla \psi
$$


In [ ]:
metric_1 = to.m(metric_0)
inv_metric_1 = to.inverse(metric_1)
volume_form_1 = to.m(volume_form_0)

left = to.divergence(to.m(phi)*to.gradient(psi, inv_metric_1, D), volume_form_1, D)
right = to.m(to.m(phi))*to.laplacian(psi, D, inv_metric_1, volume_form_1) + to.m(to.contraction(to.lower_index(metric_1, to.gradient(phi, inv_metric_1, D)), to.gradient(psi, inv_metric_1, D), [[None], [0]], [[0], [None]]))

print(f"MAE = {mae(left, right)}")

MAE = 3.2233563729278103e-07


$$\nabla \cdot \big[ \nabla\mathbf{A} + (\nabla\mathbf{A})^\textsf{T} \big] = \nabla^2\mathbf{A} + \nabla ( \nabla \cdot \mathbf{A} )$$




In [ ]:
metric_1 = to.m(metric_0)
inv_metric_1 = to.inverse(metric_1)
volume_form_1 = to.m(volume_form_0)
e_1 = to.m(e_0)
e_2 = to.m(e_1)
volume_form_2 = to.m(volume_form_1)
inv_metric_2 = to.inverse(to.m(metric_1))

gradA1 = to.partial_derivative(A, D)
gradA1_T = to.raise_index(inv_metric_1, to.transpose(to.lower_index(metric_1, gradA1)), covariant_index=gradA1.contravariant[0])
gradA1_sym_raised = to.raise_index(inv_metric_1, gradA1 + gradA1_T)
scaled = volume_form_1 * gradA1_sym_raised
dscaled = to.partial_derivative(scaled, D)
left = to.self_contraction(dscaled, [1], [0]) / to.m(volume_form_1)

right = (to.gradient(to.divergence(A, volume_form_0, D), inv_metric_2, D) - to.curl(to.curl(A, metric_0, e_1, volume_form_1, D), metric_1, e_2, volume_form_2, D)
         + to.gradient(to.divergence(A, volume_form_0, D), inv_metric_2, D))

print(f"MAE = {mae(left, right)}")

MAE = 6.278833097894676e-06


$$\psi\nabla^2\phi - \phi\nabla^2\psi = \nabla \cdot \left(\psi\nabla\phi - \phi\nabla\psi\right)$$

In [ ]:
inv_metric_1 = to.inverse(to.m(metric_0))
volume_form_1 = to.m(volume_form_0)

left = to.m(to.m(psi))*to.laplacian(phi, D, inv_metric_1, volume_form_1) - to.m(to.m(phi))*to.laplacian(psi, D, inv_metric_1, volume_form_1)
right = to.divergence(to.m(psi)*to.gradient(phi, inv_metric_1, D) - to.m(phi)*to.gradient(psi, inv_metric_1, D), volume_form_1, D)

print(f"MAE = {mae(left, right)}")

MAE = 1.1548399925231934e-06


$$\nabla^2(\phi\psi) = \phi\nabla^2\psi + 2(\nabla\phi) \cdot(\nabla\psi) + \left(\nabla^2\phi\right)\psi$$

In [ ]:
metric_1 = to.m(metric_0)
inv_metric_1 = to.inverse(metric_1)
volume_form_1 = to.m(volume_form_0)

left = to.laplacian(phi*psi, D, inv_metric_1, volume_form_1)
right = (to.m(to.m(phi))*to.laplacian(psi, D, inv_metric_1, volume_form_1)
         + 2*to.m(to.contraction(to.lower_index(metric_1, to.gradient(phi, inv_metric_1, D)), to.gradient(psi, inv_metric_1, D), [[None], [0]], [[0], [None]]))
         + to.laplacian(phi, D, inv_metric_1, volume_form_1)*to.m(to.m(psi)))

print(f"MAE = {mae(left, right)}")

MAE = 4.202127456665039e-06


$$\nabla^2(\psi\mathbf{A}) = \mathbf{A}\nabla^2\psi + 2(\nabla\psi \cdot \nabla)\mathbf{A} + \psi\nabla^2\mathbf{A}$$

In [ ]:
metric_1 = to.m(metric_0)
inv_metric_1 = to.inverse(metric_1)
e_1 = to.m(e_0)
volume_form_1 = to.m(volume_form_0)
e_2 = to.m(e_1)
volume_form_2 = to.m(volume_form_1)
inv_metric_2 = to.inverse(to.m(metric_1))

# laplacian() only supports scalar fields, so the vector Laplacian of psi*A is
# expressed via grad(div) - curl(curl), the general (curvilinear-correct)
# definition, same as in the "grad(div A) - curl(curl A) = laplacian(A)" cell.
left = to.gradient(to.divergence(psi*A, volume_form_0, D), inv_metric_2, D) - to.curl(to.curl(psi*A, metric_0, e_1, volume_form_1, D), metric_1, e_2, volume_form_2, D)
right = (to.m(to.m(A))*to.laplacian(psi, D, inv_metric_1, volume_form_1)
         + 2*to.advection(to.gradient(psi, inv_metric_1, D), to.m(A), D)
         + to.m(to.m(psi))*(to.gradient(to.divergence(A, volume_form_0, D), inv_metric_2, D) - to.curl(to.curl(A, metric_0, e_1, volume_form_1, D), metric_1, e_2, volume_form_2, D)))

print(f"MAE = {mae(left, right)}")

MAE = 9.087008038477506e-06


$$
\nabla^2 (\mathbf{A} \cdot \mathbf{B}) = 
\mathbf{A} \cdot \nabla^2 \mathbf{B} - 
\mathbf{B} \cdot \nabla^2 \mathbf{A} + 
2 \nabla \cdot \Big( (\mathbf{B} \cdot \nabla) \mathbf{A} + \mathbf{B} \times (\nabla \times \mathbf{A}) \Big)
$$


In [ ]:
metric_1 = to.m(metric_0)
inv_metric_1 = to.inverse(metric_1)
e_1 = to.m(e_0)
volume_form_1 = to.m(volume_form_0)
E_1 = to.m(E_0)
metric_2 = to.m(metric_1)
inv_metric_2 = to.inverse(metric_2)
e_2 = to.m(e_1)
volume_form_2 = to.m(volume_form_1)

# laplacian() only supports scalar fields, so the vector Laplacians of A and B
# are expressed via grad(div) - curl(curl), as above.
vecLapA = to.gradient(to.divergence(A, volume_form_0, D), inv_metric_2, D) - to.curl(to.curl(A, metric_0, e_1, volume_form_1, D), metric_1, e_2, volume_form_2, D)
vecLapB = to.gradient(to.divergence(B, volume_form_0, D), inv_metric_2, D) - to.curl(to.curl(B, metric_0, e_1, volume_form_1, D), metric_1, e_2, volume_form_2, D)

left = to.laplacian(to.contraction(to.lower_index(metric_0, A), B, [[None], [0]], [[0], [None]]), D, inv_metric_1, volume_form_1)
right = (to.contraction(to.lower_index(metric_2, to.m(to.m(A))), vecLapB, [[None], [0]], [[0], [None]])
         - to.contraction(to.lower_index(metric_2, to.m(to.m(B))), vecLapA, [[None], [0]], [[0], [None]])
         + 2*to.divergence(to.advection(B, A, D) + to.raise_index(inv_metric_1, to.cross_prod(to.m(B), to.curl(A, metric_0, e_1, volume_form_1, D), E_1)), volume_form_1, D))

print(f"MAE = {mae(left, right)}")

MAE = 0.002572706202045083


# **Third derivatives**

$$\nabla^2(\nabla\psi) =  \nabla(\nabla \cdot (\nabla\psi)) = \nabla\left(\nabla^2\psi\right)$$

$$\nabla^2(\nabla \cdot \mathbf{A}) =  \nabla \cdot (\nabla(\nabla \cdot \mathbf{A})) = \nabla \cdot \left(\nabla^2\mathbf{A}\right)$$

$$\nabla^{2}(\nabla\times\mathbf{A}) = -\nabla \times (\nabla \times (\nabla \times \mathbf{A})) = \nabla \times \left(\nabla^2\mathbf{A}\right)$$


In [ ]:
metric_1 = to.m(metric_0)
inv_metric_1 = to.inverse(metric_1)
volume_form_1 = to.m(volume_form_0)
e_1 = to.m(e_0)
e_2 = to.m(e_1)
volume_form_2 = to.m(volume_form_1)
metric_2 = to.m(metric_1)
e_3 = to.m(e_2)
volume_form_3 = to.m(volume_form_2)
inv_metric_3 = to.inverse(to.m(metric_2))

gradpsi1 = to.gradient(psi, inv_metric_1, D)
left = to.gradient(to.divergence(gradpsi1, volume_form_1, D), inv_metric_3, D) - to.curl(to.curl(gradpsi1, metric_1, e_2, volume_form_2, D), metric_2, e_3, volume_form_3, D)
center = to.gradient(to.divergence(to.gradient(psi, inv_metric_1, D), volume_form_1, D), inv_metric_3, D)
right = to.gradient(to.laplacian(psi, D, inv_metric_1, volume_form_1), inv_metric_3, D)

print(f"MAE = {mae(left, right)}")
print(f"MAE = {mae(right, center)}")
print(f"MAE = {mae(center, left)}")

MAE = 7.2451862251909915e-06
MAE = 0.0
MAE = 7.2451862251909915e-06


In [ ]:
metric_1 = to.m(metric_0)
e_1 = to.m(e_0)
volume_form_1 = to.m(volume_form_0)
metric_2 = to.m(metric_1)
e_2 = to.m(e_1)
volume_form_2 = to.m(volume_form_1)
inv_metric_2 = to.inverse(metric_2)

divA = to.divergence(A, volume_form_0, D)
left = to.laplacian(divA, D, inv_metric_2, volume_form_2)
center = to.divergence(to.gradient(divA, inv_metric_2, D), volume_form_2, D)
vecLapA = to.gradient(to.divergence(A, volume_form_0, D), inv_metric_2, D) - to.curl(to.curl(A, metric_0, e_1, volume_form_1, D), metric_1, e_2, volume_form_2, D)
right = to.divergence(vecLapA, volume_form_2, D)

print(f"MAE = {mae(left, right)}")
print(f"MAE = {mae(right, center)}")
print(f"MAE = {mae(center, left)}")

MAE = 3.5662030768435216e-06
MAE = 3.5662030768435216e-06
MAE = 0.0


In [ ]:
metric_1 = to.m(metric_0)
e_1 = to.m(e_0)
volume_form_1 = to.m(volume_form_0)
metric_2 = to.m(metric_1)
e_2 = to.m(e_1)
volume_form_2 = to.m(volume_form_1)
inv_metric_2 = to.inverse(metric_2)
e_3 = to.m(e_2)
volume_form_3 = to.m(volume_form_2)
inv_metric_3 = to.inverse(to.m(metric_2))

curlA = to.curl(A, metric_0, e_1, volume_form_1, D)
left = to.gradient(to.divergence(curlA, volume_form_1, D), inv_metric_3, D) - to.curl(to.curl(curlA, metric_1, e_2, volume_form_2, D), metric_2, e_3, volume_form_3, D)
center = -to.curl(to.curl(to.curl(A, metric_0, e_1, volume_form_1, D), metric_1, e_2, volume_form_2, D), metric_2, e_3, volume_form_3, D)
vecLapA = to.gradient(to.divergence(A, volume_form_0, D), inv_metric_2, D) - to.curl(to.curl(A, metric_0, e_1, volume_form_1, D), metric_1, e_2, volume_form_2, D)
right = to.curl(vecLapA, metric_2, e_3, volume_form_3, D)

print(f"MAE = {mae(left, right)}")
print(f"MAE = {mae(right, center)}")
print(f"MAE = {mae(center, left)}")

MAE = 1.0097732229041867e-05
MAE = 1.9646704458864406e-06
MAE = 9.122461051447317e-06
